Project Number: BYK-BUG2

Laser Audio Surveillance Device
submitted to the Faculty of WORCESTER POLYTECHNIC INSTITUTE
by Vincent Amendolare Wade Sarraf on December 19, 2005
Approved: Professor Brian King, Advisor 

https://digital.wpi.edu/pdfviewer/sb3979712

7.3dsPIC Code

In [ ]:
#include "p30f4013.h"    //support for the dsPIC

externvoid square_wave(void); //assembly language squarewave
int main(void) // begin 
{

ADC_Init(); //initialize the ADC 
TRISC=0; //set PORTC to all outputs
TRISD=0; //set PORTD to all outputs
TRISF=0; //set PORTF to all outputs 
PORTF=47;   // initialize PORTF 
PORTD=2; //sets the reset signal to high meaning no reset

while(1) //infinite loop
{

    square_wave(); //call the assembly language squarewave 
function 

} // while loop 
} // end main

In [ ]:
#include "p30f4013.h" 

         .text

         .global _square_wave
_square_wave:

main:

nop   ;actually 36 nop on either side but it would not fit
nop   ;in report that way
nop 
NEG PORTF
nop
nop
nop

goto main

return
 .end 

In [ ]:
#include "p30f4013.h" // support for dsPIC
/*Declarations*/

int i=0; // index
volatileunsignedint * iPtr;  //pointer for ADC

/*Dynamic Interupt Adjust Variables*/
int high_time=0;
int low_time=0;
int delay=0;

/*Smooth Adjust Variables*/
int past = 0; 
int present = 0;
int smooth = 0; 

//Prototypes 

void ADC_Init(void); 
void __attribute__((__interrupt__)) _ADCInterrupt(void);

void ADC_Init(void)
{
// Configure Analog Pins

     TRISB = 1; // sets 1st pin in PORTB(AN0) to an input
     ADPCFG = 0xFFFB; /*sets AN2 to an analog input,
                           the      rest      are      digital      I/O

// Selecting Input to S/H Channels 

    ADCHS = 2;   // 0 set AN0 as input to CH0
                        // CH0+ input is AN2.
                        // CHO- input is VREFL (AVss)
    ADCSSL = 0;  // no inputs are scanned

// Set PWM Frequency 

      TMR1 = 0x0000;     // timer 1 default settings
      PR1 = 2105;        // PWM period 
      T1CON = 0x8000;  // turn on timer 1

// Set PWM Duty Cycle Timer

      TMR2 = 0x0000;     // timer 2 default settings 
      PR2 = 0x000;       // Duty Cycle 
      T2CON = 0x8000;  // turn on timer 2 

// Select ADC Conversion Clock

      TMR3 = 0x0000;     // timer 3 default settings
      PR3 = 8192;        // Sampling Period
      T3CON = 0x8000;  // turn on timer 3 

In [ ]:
// Select ADC Conversion Trigger 

      ADCON1bits.ADSIDL = 0; // continue operation in idle mode
      ADCON1bits.FORM = 0; // sets output of ADC as int
      ADCON1bits.SSRC = 2; // timer 3 is sample/convert trigger
      ADCON1bits.ASAM = 1; // sample once convert is done
      ADCON2bits.VCFG = 3;  // AVdd and AVss are voltage 
      ADCON2bits.CSCNA = 0; // Do not scan input selections 
      ADCON2bits.SMPI = 0; // 1 sample per interrupt 
      CON2bits.BUFM      =      0;      // 16 word buffers
      ADCON2bits.ALTS = 0; // Use A inputs for multiplexing
      ADCON3bits.ADRC = 0; // sets clock as system derived
      ADCON3bits.ADCS = 32; // adc conversion clock select bit 

// turn ADC on

      ADCON1bits.ADON = 1;

// Enable Interrupts

      IFS0bits.ADIF = 0;       // clear conversion interrupt
      IEC0bits.ADIE = 1;       // enable ADC interrupt 
      IFS0bits.T3IF = 0;       // clear timer 3 interrupt
      IEC0bits.T3IE = 0;       // enable timer 3 interrupt
      IFS0bits.T2IF = 0;       // clear timer 2 interrupt 
      IEC0bits.T2IE = 1;       // enable timer 2 interrupt
      IFS0bits.T1IF = 0;       // clear timer 1 interrupt
      IEC0bits.T1IE = 1;       // enable timer 1 interrupt

      return;
}

In [ ]:
void __attribute__((__interrupt__)) _ADCInterrupt(void)
{
      PR2 += smooth;     // 4th addition of the smooth factor 
      PORTC = 0x2FFF;    // set PWM high
      ORTF=47;                  // inverted clock high
      TMR3 = 0;          // resets Timer 3
      IFS0bits.T3IF = 0; // clear interrupt flag
      PORTD = 0;         // begin reset signal
      past = present;    // save last pulse width 
      iPtr = &ADCBUF0; // set pointer to ADCBUF0
      present = (*iPtr>>1);  // load new duty cycle into PWM

      smooth = (present-past)>>2; //calculate new smooth factor 

/* to accommodate delays from branching executing
instructions the floor value is 36 but we still
need to represent the values between 1 and 36*/ 

     if(PR2<36){PR2+=36;}

/* past values for pwm high time are between 36
and 2048 this scales down the high time to a max of 256*/

      high_time = past>>3;
      low_time = 256-high_time;

/* this for loop is timed up to last as long as the next
pulse's high time would if there were no interrupt
wasting time is avoided by making continuing to modulate port F */
      for(i=0; i<high_time; i++){
            PORTF=-PORTF;}

      PORTC = 0; // set PWM low after appropriate delay

/* this delay makes the Interrupt last as long
as any of other PWM high/low periods*/

      for(i=0; i<low_time; i++){
            ORTF=-PORTF;}            // modulate port F
            PORTD=2;                  // end reset signal
            IEC0bits.ADIE = 1; // enable ADC interrupt
            IFS0bits.ADIF = 0; // Clear the A/D Interrupt flag
            TMR1=TMR3+4;            // sync up timers

            delay=1;                  // delays for timing
            delay=1; 
            delay++; 
            PORTC=0x2FFF;            // set PWM high 
}

In [ ]:
void __attribute__((__interrupt__, __shadow__)) _T2Interrupt(void)
{
                PORTC      =      0;                        // Set PWM Low 
                IFS0bits.T2IF = 0;       //clear interrupt flag
                0bits.T2IE      =      0;            // disable timer 2 
interrupt
}

void __attribute__((__interrupt__, __shadow__)) _T1Interrupt(void)
{
            PORTC      =      0x2FFF;                  // Set PWM High
            PR2      +=      smooth;                  // increment pulse width

/* trying to sync up timer 2 and 1 time to will not
increment during this command which takes two clock periods
it also does not increment when t2 interrupt flag
is cleared so we need to add 3*/

            TMR2      =      TMR1+

            if (PR2<36){PR2=36;} // avoid underflow 
            IFS0bits.T1IF = 0;  // clear interrupt flag 
            IFS0bits.T2IF      =      0;      // Clear timer 2 interrupt
            IEC0bits.T2IE      =      1;      // Enable timer 2 interrupt
